# 01 - Tien xu ly du lieu anh & video (Face Recognition Security System)

**Muc tieu:** xay dung pipeline tien xu ly hoan chinh truoc khi sang buoc detection + recognition:
1. Tai dataset mo phong gallery nhieu danh tinh (LFW)
2. Face detection + alignment chuan ArcFace 112x112 (RetinaFace)
3. Loc chat luong (anh mo, mat qua nho)
4. Chia Gallery (enrollment) / Probe (test) theo tung nguoi
5. Trich khung hinh tu video giam sat (nhieu nguoi trong khung)
6. Data augmentation cho gallery

**Dataset duoc chon: LFW (Labeled Faces in the Wild)** - tai truc tiep qua `sklearn`, khong can link thu cong, co nhieu anh/nguoi de mo phong gallery/probe. Khi trien khai that, thay `data/raw/lfw` bang anh enrollment that cua nguoi duoc cap quyen.

In [ ]:
# ===== CELL 1: Mount Google Drive & tao cau truc thu muc =====
from google.colab import drive
drive.mount('/content/drive')

import os

BASE_DIR = '/content/drive/MyDrive/SecurityFaceSystem'
SUBFOLDERS = [
    'data/raw/lfw',
    'data/processed/gallery',
    'data/splits/gallery',
    'data/splits/probe',
    'data/frames',
    'data/augmented',
    'models',
    'results',
    'notebooks',
]

os.makedirs(BASE_DIR, exist_ok=True)
for f in SUBFOLDERS:
    os.makedirs(os.path.join(BASE_DIR, f), exist_ok=True)

print("Cau truc thu muc da tao tai:", BASE_DIR)
for f in SUBFOLDERS:
    print("   -", f)


In [ ]:
# ===== CELL 2: Cai dat thu vien =====
!pip install -q retina-face opencv-python-headless albumentations

print("Da cai: retina-face (RetinaFace detector), opencv-python-headless, albumentations")
print("Luu y: retina-face dung TensorFlow backend, chay song song binh thuong voi PyTorch (YOLOv8) o notebook sau, khong xung dot.")


In [ ]:
# ===== CELL 3: Import & set seed =====
import json
import random
import shutil
import cv2
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from sklearn.datasets import fetch_lfw_people
from sklearn.model_selection import train_test_split
from retinaface import RetinaFace

SEED = 42
random.seed(SEED)
np.random.seed(SEED)

RAW_DIR       = os.path.join(BASE_DIR, 'data/raw/lfw')
PROCESSED_DIR = os.path.join(BASE_DIR, 'data/processed/gallery')
SPLIT_GALLERY = os.path.join(BASE_DIR, 'data/splits/gallery')
SPLIT_PROBE   = os.path.join(BASE_DIR, 'data/splits/probe')
FRAMES_DIR    = os.path.join(BASE_DIR, 'data/frames')
AUG_DIR       = os.path.join(BASE_DIR, 'data/augmented')

print(f"Seed = {SEED}")
print(f"RAW_DIR       = {RAW_DIR}")
print(f"PROCESSED_DIR = {PROCESSED_DIR}")


## Buoc 1 - Tai dataset (mo phong gallery nhieu danh tinh)

Dung **LFW** qua `sklearn.datasets.fetch_lfw_people`:
- Tai tu dong, khong can link thu cong.
- Chon `min_faces_per_person=20` de moi danh tinh co du anh cho viec chia gallery/probe.
- **Luu y quan trong:** day la dataset de dung va kiem thu pipeline tien xu ly ngay lap tuc. Khi trien khai that, thu muc `data/raw/lfw` nay can duoc thay bang anh dang ky (enrollment) cua nhung nguoi duoc cap quyen that trong he thong.

In [ ]:
# ===== CELL 4: Tai LFW & luu thanh cau truc folder theo danh tinh =====
print("Dang tai LFW (lan dau se mat vai phut)...")
lfw = fetch_lfw_people(min_faces_per_person=20, resize=1.0, color=True, download_if_missing=True)

images       = lfw.images          # (N, H, W, 3), gia tri float 0-255
labels       = lfw.target
target_names = lfw.target_names

print(f"Tong so anh: {images.shape[0]}")
print(f"So danh tinh (nguoi): {len(target_names)}")
print(f"Kich thuoc anh goc: {images.shape[1]}x{images.shape[2]}")

# Luu ra file anh that theo cau truc data/raw/lfw/<ten_nguoi>/img_XXX.jpg
saved_count = 0
for idx in range(images.shape[0]):
    person_name = target_names[labels[idx]].replace(' ', '_')
    person_dir = os.path.join(RAW_DIR, person_name)
    os.makedirs(person_dir, exist_ok=True)

    img_uint8 = np.clip(images[idx], 0, 255).astype(np.uint8)
    img_bgr = cv2.cvtColor(img_uint8, cv2.COLOR_RGB2BGR)

    existing = len(os.listdir(person_dir))
    save_path = os.path.join(person_dir, f'img_{existing:03d}.jpg')
    cv2.imwrite(save_path, img_bgr)
    saved_count += 1

print(f"Da luu {saved_count} anh goc vao {RAW_DIR}")


In [ ]:
# ===== CELL 5: Xem thu vai anh goc =====
sample_people = os.listdir(RAW_DIR)[:5]
fig, axes = plt.subplots(1, len(sample_people), figsize=(15, 3))
for ax, person in zip(axes, sample_people):
    person_dir = os.path.join(RAW_DIR, person)
    img_path = os.path.join(person_dir, os.listdir(person_dir)[0])
    img = cv2.cvtColor(cv2.imread(img_path), cv2.COLOR_BGR2RGB)
    ax.imshow(img)
    ax.set_title(person, fontsize=9)
    ax.axis('off')
plt.tight_layout()
plt.show()


## Buoc 2 - Face Detection + Alignment (RetinaFace, chuan ArcFace 112x112)

Moi anh duoc:
1. Phat hien khuon mat + 5 diem landmark (mat trai, mat phai, mui, khoe mieng trai/phai) bang RetinaFace.
2. Can chinh (align) bang affine transform ve dung template chuan 112x112 ma ArcFace mong doi - buoc nay cuc ky quan trong, vi ArcFace duoc huan luyen tren anh da align, bo qua buoc nay se lam giam manh do chinh xac nhan dang.
3. Loc chat luong: bo anh mo (Laplacian variance) va khuon mat qua nho.

In [ ]:
# ===== CELL 6: Chuan 5-diem ArcFace (112x112) & ham align =====

# Template chuan 112x112 dung trong ArcFace/InsightFace
# (thu tu: right_eye, left_eye, nose, mouth_right, mouth_left)
ARCFACE_TEMPLATE_112 = np.array([
    [73.5318, 51.5014],   # right_eye
    [38.2946, 51.6963],   # left_eye
    [56.0252, 71.7366],   # nose
    [70.7299, 92.2041],   # mouth_right
    [41.5493, 92.3655],   # mouth_left
], dtype=np.float32)

TARGET_SIZE = (112, 112)


def align_face(img_bgr, landmarks):
    """Can chinh khuon mat ve 112x112 theo landmark chuan ArcFace."""
    src_pts = np.array([
        landmarks['right_eye'],
        landmarks['left_eye'],
        landmarks['nose'],
        landmarks['mouth_right'],
        landmarks['mouth_left'],
    ], dtype=np.float32)

    M, _ = cv2.estimateAffinePartial2D(src_pts, ARCFACE_TEMPLATE_112, method=cv2.LMEDS)
    if M is None:
        return None
    aligned = cv2.warpAffine(img_bgr, M, TARGET_SIZE, borderValue=0.0)
    return aligned


def detect_and_align(img_bgr):
    """Phat hien khuon mat co score cao nhat trong anh va tra ve anh da align + bbox."""
    faces = RetinaFace.detect_faces(img_bgr)
    if not isinstance(faces, dict) or len(faces) == 0:
        return None, None

    best_key = max(faces, key=lambda k: faces[k]['score'])
    face_data = faces[best_key]

    aligned = align_face(img_bgr, face_data['landmarks'])
    return aligned, face_data['facial_area']

print("Da dinh nghia align_face() va detect_and_align()")


In [ ]:
# ===== CELL 7: Ham kiem tra chat luong anh =====

def is_blurry(img_bgr, threshold=100.0):
    gray = cv2.cvtColor(img_bgr, cv2.COLOR_BGR2GRAY)
    lap_var = cv2.Laplacian(gray, cv2.CV_64F).var()
    return lap_var < threshold, lap_var


def is_face_too_small(facial_area, min_size=40):
    x1, y1, x2, y2 = facial_area
    w, h = x2 - x1, y2 - y1
    return min(w, h) < min_size

print("Da dinh nghia is_blurry() va is_face_too_small()")
print("Voi anh gallery (enrollment): ap dung loc NGHIEM NGAT - anh mo/mat nho nen bi loai vi se lam hong embedding.")
print("Voi frame trich tu video giam sat: chi NEN GAN NHAN canh bao, khong loai bo, vi he thong van can co nhan dang ca nguoi o xa.")


In [ ]:
# ===== CELL 8: Vong lap tien xu ly chinh (raw -> processed, co loc chat luong) =====

stats = {'total': 0, 'no_face': 0, 'too_blurry': 0, 'too_small': 0, 'kept': 0}
log_rows = []

for person in sorted(os.listdir(RAW_DIR)):
    person_raw_dir = os.path.join(RAW_DIR, person)
    person_out_dir = os.path.join(PROCESSED_DIR, person)
    os.makedirs(person_out_dir, exist_ok=True)

    for fname in sorted(os.listdir(person_raw_dir)):
        stats['total'] += 1
        img_path = os.path.join(person_raw_dir, fname)
        img_bgr = cv2.imread(img_path)

        aligned, facial_area = detect_and_align(img_bgr)

        if aligned is None:
            stats['no_face'] += 1
            log_rows.append([person, fname, 'no_face'])
            continue

        if facial_area is not None and is_face_too_small(facial_area, min_size=40):
            stats['too_small'] += 1
            log_rows.append([person, fname, 'too_small'])
            continue

        blurry, lap_var = is_blurry(aligned, threshold=80.0)
        if blurry:
            stats['too_blurry'] += 1
            log_rows.append([person, fname, f'too_blurry (lap_var={lap_var:.1f})'])
            continue

        out_path = os.path.join(person_out_dir, fname)
        cv2.imwrite(out_path, aligned)
        stats['kept'] += 1
        log_rows.append([person, fname, 'kept'])

print("="*50)
print("KET QUA TIEN XU LY")
print("="*50)
for k, v in stats.items():
    print(f"{k:12s}: {v}")

log_df = pd.DataFrame(log_rows, columns=['person', 'filename', 'status'])
log_df.to_csv(os.path.join(BASE_DIR, 'data', 'preprocessing_log.csv'), index=False)
print(f"\nLog chi tiet luu tai: data/preprocessing_log.csv")


In [ ]:
# ===== CELL 9: Xem thu anh da align =====
sample_people = [p for p in os.listdir(PROCESSED_DIR) if os.listdir(os.path.join(PROCESSED_DIR, p))][:5]
fig, axes = plt.subplots(1, len(sample_people), figsize=(12, 3))
for ax, person in zip(axes, sample_people):
    person_dir = os.path.join(PROCESSED_DIR, person)
    img_path = os.path.join(person_dir, os.listdir(person_dir)[0])
    img = cv2.cvtColor(cv2.imread(img_path), cv2.COLOR_BGR2RGB)
    ax.imshow(img)
    ax.set_title(f"{person}\n112x112 aligned", fontsize=8)
    ax.axis('off')
plt.tight_layout()
plt.show()


## Buoc 3 - Chia Gallery (enrollment) / Probe (test) theo tung danh tinh

Moi nguoi duoc chia theo ti le 70% Gallery (anh dung de dang ky / tao embedding chuan) - 30% Probe (anh dung de test he thong co nhan ra dung nguoi khong). Chi giu nhung nguoi co >= 4 anh sau khi loc chat luong, de dam bao moi phan chia co du anh.

In [ ]:
# ===== CELL 10: Chia Gallery / Probe =====

MIN_IMAGES_PER_PERSON = 4
manifest_rows = []

for person in sorted(os.listdir(PROCESSED_DIR)):
    person_dir = os.path.join(PROCESSED_DIR, person)
    images_list = sorted(os.listdir(person_dir))

    if len(images_list) < MIN_IMAGES_PER_PERSON:
        continue

    gallery_imgs, probe_imgs = train_test_split(
        images_list, test_size=0.3, random_state=SEED
    )

    gallery_person_dir = os.path.join(SPLIT_GALLERY, person)
    probe_person_dir = os.path.join(SPLIT_PROBE, person)
    os.makedirs(gallery_person_dir, exist_ok=True)
    os.makedirs(probe_person_dir, exist_ok=True)

    for fname in gallery_imgs:
        shutil.copy(os.path.join(person_dir, fname), os.path.join(gallery_person_dir, fname))
        manifest_rows.append([person, fname, 'gallery'])

    for fname in probe_imgs:
        shutil.copy(os.path.join(person_dir, fname), os.path.join(probe_person_dir, fname))
        manifest_rows.append([person, fname, 'probe'])

manifest_df = pd.DataFrame(manifest_rows, columns=['person', 'filename', 'split'])
manifest_df.to_csv(os.path.join(BASE_DIR, 'data', 'split_manifest.csv'), index=False)

n_people = manifest_df['person'].nunique()
print(f"So danh tinh du dieu kien (>={MIN_IMAGES_PER_PERSON} anh): {n_people}")
print(f"Gallery: {len(manifest_df[manifest_df.split=='gallery'])} anh")
print(f"Probe:   {len(manifest_df[manifest_df.split=='probe'])} anh")
print(f"Manifest luu tai: data/split_manifest.csv")


## Buoc 4 - Trich khung hinh tu video giam sat (nhieu nguoi trong khung hinh)

Vi day la du lieu dac thu cua nhom (video quay bang dien thoai / camera that), Claude khong the nhung san video mau. Hay:
1. Quay 1 video ngan (30s-1 phut) co nhieu nguoi di lai, co nguoi o xa/gan khac nhau.
2. Upload len Google Drive, dat duong dan vao bien `VIDEO_PATH` ben duoi.

Ham `extract_frames()` se lay mau theo FPS mong muon (khong lay toan bo khung hinh de tranh du thua gan trung nhau), dong thoi gan nhan canh bao (khong loai bo) nhung frame bi mo.

In [ ]:
# ===== CELL 11: Ham trich khung hinh tu video =====

def extract_frames(video_path, output_dir, target_fps=2, blur_threshold=60.0):
    os.makedirs(output_dir, exist_ok=True)
    cap = cv2.VideoCapture(video_path)

    if not cap.isOpened():
        print(f"Khong mo duoc video: {video_path}")
        return []

    source_fps = cap.get(cv2.CAP_PROP_FPS) or 25
    frame_interval = max(1, int(round(source_fps / target_fps)))

    frame_idx = 0
    saved_idx = 0
    log = []

    while True:
        ret, frame = cap.read()
        if not ret:
            break

        if frame_idx % frame_interval == 0:
            timestamp_sec = frame_idx / source_fps
            blurry, lap_var = is_blurry(frame, threshold=blur_threshold)
            quality_flag = 'blurry' if blurry else 'ok'

            fname = f'frame_{saved_idx:05d}_t{timestamp_sec:.1f}s.jpg'
            cv2.imwrite(os.path.join(output_dir, fname), frame)
            log.append([fname, timestamp_sec, quality_flag, round(lap_var, 1)])
            saved_idx += 1

        frame_idx += 1

    cap.release()

    log_df = pd.DataFrame(log, columns=['filename', 'timestamp_sec', 'quality', 'blur_score'])
    log_path = os.path.join(output_dir, '_frames_log.csv')
    log_df.to_csv(log_path, index=False)

    print(f"Da trich {saved_idx} khung hinh (target_fps={target_fps}) tu {video_path}")
    print(f"So khung hinh bi gan nhan 'blurry': {(log_df.quality=='blurry').sum()}")
    print(f"Log luu tai: {log_path}")
    return log

print("Da dinh nghia extract_frames()")


In [ ]:
# ===== CELL 12: Chay thu trich khung hinh (thay VIDEO_PATH bang video that cua nhom) =====

VIDEO_PATH = os.path.join(BASE_DIR, 'data', 'sample_security_video.mp4')  # <-- doi thanh duong dan video that

if os.path.exists(VIDEO_PATH):
    video_frames_dir = os.path.join(FRAMES_DIR, os.path.splitext(os.path.basename(VIDEO_PATH))[0])
    extract_frames(VIDEO_PATH, video_frames_dir, target_fps=2)

    sample_frames = sorted(os.listdir(video_frames_dir))
    sample_frames = [f for f in sample_frames if f.endswith('.jpg')][:5]
    fig, axes = plt.subplots(1, len(sample_frames), figsize=(15, 3))
    for ax, fname in zip(axes, sample_frames):
        img = cv2.cvtColor(cv2.imread(os.path.join(video_frames_dir, fname)), cv2.COLOR_BGR2RGB)
        ax.imshow(img)
        ax.axis('off')
    plt.tight_layout()
    plt.show()
else:
    print(f"Chua tim thay video tai {VIDEO_PATH}")
    print("Upload video test dong nguoi vao duong dan tren (hoac doi bien VIDEO_PATH), roi chay lai cell nay.")


## Buoc 5 (tuy chon) - Data Augmentation cho Gallery

Vi anh enrollment thuc te (dang ky nhan vien/nguoi duoc phep) thuong it (co the chi 3-5 anh/nguoi), augmentation giup gallery day hon, robust hon voi dieu kien anh sang/goc quay khac cua camera an ninh thuc te.

In [ ]:
# ===== CELL 13: Augmentation cho anh gallery =====
import albumentations as A

augment_pipeline = A.Compose([
    A.RandomBrightnessContrast(brightness_limit=0.25, contrast_limit=0.25, p=0.8),
    A.HorizontalFlip(p=0.5),
    A.Rotate(limit=8, border_mode=cv2.BORDER_CONSTANT, p=0.6),
    A.GaussNoise(var_limit=(5.0, 20.0), p=0.3),
])

N_AUG_PER_IMAGE = 3
aug_count = 0

for person in sorted(os.listdir(SPLIT_GALLERY)):
    person_dir = os.path.join(SPLIT_GALLERY, person)
    aug_person_dir = os.path.join(AUG_DIR, person)
    os.makedirs(aug_person_dir, exist_ok=True)

    for fname in os.listdir(person_dir):
        img = cv2.imread(os.path.join(person_dir, fname))
        for i in range(N_AUG_PER_IMAGE):
            augmented = augment_pipeline(image=img)['image']
            aug_fname = f"{os.path.splitext(fname)[0]}_aug{i}.jpg"
            cv2.imwrite(os.path.join(aug_person_dir, aug_fname), augmented)
            aug_count += 1

print(f"Da sinh them {aug_count} anh augmented (x{N_AUG_PER_IMAGE}/anh goc) vao {AUG_DIR}")
print("Dung data/splits/gallery (goc) + data/augmented cho buoc xay gallery embedding o notebook tiep theo.")


## Tong ket Buoc Tien xu ly

Ket qua buoc nay:
- `data/raw/lfw/` - anh goc theo tung danh tinh (thay bang anh enrollment that khi trien khai)
- `data/processed/gallery/` - anh da align 112x112, da loc chat luong
- `data/splits/gallery/` & `data/splits/probe/` - chia theo tung nguoi (70/30)
- `data/augmented/` - anh gallery da tang cuong
- `data/frames/` - khung hinh trich tu video giam sat
- `data/preprocessing_log.csv`, `data/split_manifest.csv` - log day du de truy vet

Buoc tiep theo: dung `data/splits/gallery` (+ `data/augmented`) de build embedding gallery bang ArcFace, sau do test nhan dang tren `data/splits/probe` va `data/frames`.

In [ ]:
# ===== CELL 14: Luu metadata tong ket =====

metadata = {
    'dataset': 'LFW (Labeled Faces in the Wild) via sklearn.fetch_lfw_people',
    'note': 'Dataset demo de dung pipeline - can thay bang anh enrollment + video giam sat that khi trien khai.',
    'seed': SEED,
    'target_face_size': TARGET_SIZE,
    'stats_preprocessing': stats,
    'n_identities_after_split': int(n_people),
    'gallery_images': int(len(manifest_df[manifest_df.split=='gallery'])),
    'probe_images': int(len(manifest_df[manifest_df.split=='probe'])),
}

metadata_path = os.path.join(BASE_DIR, 'data', 'dataset_metadata.json')
with open(metadata_path, 'w', encoding='utf-8') as f:
    json.dump(metadata, f, ensure_ascii=False, indent=2)

print("="*50)
print("TIEN XU LY DU LIEU HOAN TAT")
print("="*50)
print(json.dumps(metadata, ensure_ascii=False, indent=2))
